# 02 - Model Prototyping

**Purpose:** compare XGBoost, LightGBM and CatBoost on the leakage-safe split,
tune the decision threshold against a cost model, and quantify what stacking and
SHAP feature selection actually buy.

Everything here calls the same `src/` modules that `main.py` runs, so results
transfer directly to the CLI pipeline.


## 1. Setup

In [1]:
"""Shared notebook setup: locate the project root, import the pipeline, set plot style."""
from pathlib import Path
import sys, os, warnings

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PipelineConfig, configure_logging
from src import preprocessing

configure_logging("INFO")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60)
print("project root:", PROJECT_ROOT)


project root: /Users/dharun/FFDUEAIAEM/financial_fraud_detection


In [2]:
"""Find the raw CSVs.

Resolution order:
1. ``FFDD_DATA_PATH`` environment variable
2. ``<project>/data/raw``
3. a few sibling folders commonly used for the Kaggle export
"""
def locate_data_dir() -> Path:
    candidates = [
        os.environ.get("FFDD_DATA_PATH"),
        PROJECT_ROOT / "data" / "raw",
        PROJECT_ROOT.parent / "Dataset",
        PROJECT_ROOT.parent / "data" / "raw",
        PROJECT_ROOT.parent,
        Path.home() / "Final year project",
        Path.home() / "final year project",
    ]
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser()
        if path.is_file():
            path = path.parent
        if path.is_dir() and any("transaction" in p.name.lower() for p in path.glob("*.csv")):
            return path
    raise FileNotFoundError("Set FFDD_DATA_PATH to the folder holding train_transaction.csv")

DATA_DIR = locate_data_dir()
print("data directory:", DATA_DIR)
print(sorted(p.name for p in DATA_DIR.glob("*.csv"))[:8])


data directory: /Users/dharun/Final year project
['sample_submission.csv', 'test_identity.csv', 'test_merged.csv', 'test_transaction.csv', 'train_identity.csv', 'train_transaction.csv']


## 2. Build the leakage-safe dataset

The split is by `TransactionDT`: the newest 20 % becomes validation. All
encoding statistics and median imputations are fitted on the earlier 80 %.

In [3]:
from src.preprocessing import build_processed_dataset, load_raw_data, select_features

SAMPLE_ROWS = 100_000   # raise to None for a full run (slower)

config = PipelineConfig(
    data_path=str(DATA_DIR),
    project_root=str(PROJECT_ROOT),
    top_n_features=30,
    shap_sample_size=10_000,
    validation_fraction=0.20,
    random_state=42,
)

transaction_path = next(p for p in sorted(DATA_DIR.glob("*.csv"))
                        if "transaction" in p.name.lower() and "train" in p.name.lower())
identity_path = next((p for p in sorted(DATA_DIR.glob("*.csv"))
                      if "identity" in p.name.lower() and "train" in p.name.lower()), None)

raw = load_raw_data(config, transaction_path=transaction_path,
                    identity_path=identity_path, load_identity=identity_path is not None)
if SAMPLE_ROWS and len(raw) > SAMPLE_ROWS:
    raw = (raw.sample(n=SAMPLE_ROWS, random_state=42)
              .sort_values("TransactionDT").reset_index(drop=True))

processed = build_processed_dataset(config, raw_df=raw, save=False)
print(processed.summary())
print("features:", processed.X_train.shape[1])


2026-11-06 14:58:25 | INFO     | src.preprocessing            | Loading transactions from /Users/dharun/Final year project/train_transaction.csv


2026-11-06 14:58:29 | INFO     | src.preprocessing            |   transactions: 590,540 x 394


2026-11-06 14:58:29 | INFO     | src.preprocessing            | Loading identity from /Users/dharun/Final year project/train_identity.csv


2026-11-06 14:58:29 | INFO     | src.preprocessing            |   identity: 144,233 x 41


2026-11-06 14:58:29 | INFO     | src.preprocessing            |   merge: 590,540 rows x 434 cols; 24.4% of transactions have identity data


2026-11-06 14:58:34 | INFO     | src.preprocessing            | reduce_mem_usage: 1984.2 MB -> 617.3 MB (68.9% reduction)


... (4 outputs left out)

2026-11-06 14:58:35 | INFO     | src.preprocessing            | Dropping 29 columns with > 80% missing


2026-11-06 14:58:36 | INFO     | src.preprocessing            | Encoding 25 categorical and 391 numeric columns with 'frequency' strategy


2026-11-06 14:58:37 | INFO     | src.preprocessing            | Preprocessing finished in 2.5s: train=(80000, 416), valid=(20000, 416), fraud rate train=3.606%, valid=3.455%


train=(80000, 416), valid=(20000, 416), fraud rate train=3.606%, valid=3.455%
features: 416


## 3. SHAP feature selection (top 30)

A fast baseline XGBoost is fitted on the training period, TreeSHAP values are
computed on a 10k-row background sample, and features are ranked by mean
absolute contribution. The validation period is never used for the ranking.

In [4]:
from src.feature_selection import run_feature_selection

selection = run_feature_selection(
    processed.X_train, processed.y_train,
    X_valid=processed.X_valid, y_valid=processed.y_valid,
    top_n=30, shap_sample_size=10_000, output_dir=PROJECT_ROOT / "outputs",
    random_state=42,
)
selection.ranking.head(20)


2026-11-06 14:58:37 | INFO     | src.feature_selection        | Fitting baseline XGBoost on 80,000 rows


2026-11-06 14:58:42 | INFO     | src.feature_selection        | Baseline hold-out performance: ROC-AUC=0.8813, PR-AUC=0.4643


2026-11-06 14:58:43 | INFO     | src.feature_selection        | Computed TreeSHAP for 10000 x 416 features in 0.6s


2026-11-06 14:58:43 | INFO     | src.feature_selection        | Selected features written to /Users/dharun/FFDUEAIAEM/financial_fraud_detection/outputs/selected_features.json


/private/tmp/ffdd_probe/final/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-11-06 14:58:45 | INFO     | src.feature_selection        | Feature selection finished in 7.6s: 416 -> 30 features. Top-10: ['C13', 'card6', 'C1', 'C14', 'TransactionAmt', 'card3', 'C11', 'card1', 'M6', 'V49']


,feature,mean_abs_shap,rank,selected
0,C13,0.184917,1,True
1,card6,0.173145,2,True
2,C1,0.163204,3,True
3,C14,0.146229,4,True
4,TransactionAmt,0.140681,5,True
5,card3,0.127531,6,True
6,C11,0.124643,7,True
7,card1,0.124550,8,True
8,M6,0.112627,9,True
9,V49,0.111931,10,True


In [5]:
fig, ax = plt.subplots(figsize=(8, 7))
top = selection.mean_abs_shap.head(20).iloc[::-1]
ax.barh(top.index, top.values, color="#1f4e79")
ax.set_xlabel("mean |SHAP value|")
ax.set_title("Top 20 features - baseline XGBoost + TreeSHAP")
fig.tight_layout()
plt.show()

selected = selection.selected_features
X_train = select_features(processed.X_train, selected)
X_valid = select_features(processed.X_valid, selected)
print(f"reduced to {X_train.shape[1]} features")


reduced to 30 features


/var/folders/89/wjdg33k579n81f57cjynwt6h0000gn/T/ipykernel_5320/2752967784.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Train the three boosters

Same features, same chronological split, imbalance-aware defaults. Early
stopping uses a slice **inside** the training period, so the validation period
stays untouched and the scores below are honest.

In [6]:
import time
from src.evaluation import evaluate_predictions
from src.models import build_model, chronological_holdout, fit_estimator

X_fit, y_fit, X_stop, y_stop = chronological_holdout(X_train, processed.y_train, fraction=0.10)

fitted, reports = {}, {}
for name in ["xgboost", "lightgbm", "catboost"]:
    model = build_model(name, y_train=y_fit, random_state=42)
    started = time.perf_counter()
    fit_estimator(model, X_fit, y_fit, X_stop=X_stop, y_stop=y_stop)
    elapsed = time.perf_counter() - started
    probabilities = model.predict_proba(X_valid)[:, 1]
    report = evaluate_predictions(
        processed.y_valid, probabilities, model_name=name,
        figures_dir=PROJECT_ROOT / "outputs" / "figures", make_figures=False,
    )
    fitted[name] = model
    reports[name] = report
    print(f"{name:<9} fit {elapsed:5.1f}s | ROC-AUC {report.metrics_at_optimal['roc_auc']:.4f} "
          f"| PR-AUC {report.metrics_at_optimal['pr_auc']:.4f} "
          f"| threshold {report.threshold_optimal:.3f}")


2026-11-06 14:58:48 | INFO     | src.evaluation               | xgboost: ROC-AUC=0.8639 PR-AUC=0.4452 F1@opt=0.1238 thr=0.040 cost=75,640


xgboost   fit   3.2s | ROC-AUC 0.8639 | PR-AUC 0.4452 | threshold 0.040


2026-11-06 14:58:54 | INFO     | src.evaluation               | lightgbm: ROC-AUC=0.8738 PR-AUC=0.4492 F1@opt=0.1127 thr=0.016 cost=71,910


lightgbm  fit   5.8s | ROC-AUC 0.8738 | PR-AUC 0.4492 | threshold 0.016


2026-11-06 14:59:08 | INFO     | src.evaluation               | catboost: ROC-AUC=0.8665 PR-AUC=0.4413 F1@opt=0.1085 thr=0.032 cost=75,775


catboost  fit  13.3s | ROC-AUC 0.8665 | PR-AUC 0.4413 | threshold 0.032


## 5. Comparison and the cost-sensitive threshold

A false negative costs `fraud_cost` (chargeback + goods); a false positive costs
`friction_cost` (support, churn). With the defaults (500 vs 5) the
expected-cost optimum is a *low* threshold: it is worth blocking many genuine
transactions to avoid missing fraud.

In [7]:
comparison = pd.DataFrame({
    name: {
        "ROC-AUC": r.metrics_at_optimal["roc_auc"],
        "PR-AUC": r.metrics_at_optimal["pr_auc"],
        "F1 @ optimum": r.metrics_at_optimal["f1"],
        "Balanced acc.": r.metrics_at_optimal["balanced_accuracy"],
        "Brier": r.metrics_at_optimal["brier"],
        "Threshold": r.threshold_optimal,
        "Recall": r.metrics_at_optimal["recall"],
        "Precision": r.metrics_at_optimal["precision"],
        "Expected cost": r.cost_optimal,
    }
    for name, r in reports.items()
}).T.sort_values("PR-AUC", ascending=False)
comparison.round(4)


,ROC-AUC,PR-AUC,F1 @ optimum,Balanced acc.,Brier,Threshold,Recall,Precision,Expected cost
lightgbm,0.8738,0.4492,0.1127,0.7059,0.0301,0.016,0.9392,0.0599,71910.0
xgboost,0.8639,0.4452,0.1238,0.7258,0.0301,0.040,0.9088,0.0664,75640.0
catboost,0.8665,0.4413,0.1085,0.6934,0.0412,0.032,0.9334,0.0576,75775.0


In [8]:
from sklearn.metrics import precision_recall_curve

y_val = processed.y_valid.to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, model in fitted.items():
    probs = model.predict_proba(X_valid)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, probs)
    ap = reports[name].metrics_at_optimal["pr_auc"]
    axes[0].plot(recall, precision, label=f"{name} (AP={ap:.3f})")
axes[0].axhline(y_val.mean(), ls="--", color="grey", label="no-skill")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precision")
axes[0].set_title("Precision-Recall on the validation period")
axes[0].legend()

grid = np.linspace(0.01, 0.99, 80)
for name, model in fitted.items():
    probs = model.predict_proba(X_valid)[:, 1]
    costs = []
    for threshold in grid:
        pred = (probs >= threshold).astype(int)
        fn = np.sum((y_val == 1) & (pred == 0))
        fp = np.sum((y_val == 0) & (pred == 1))
        costs.append(fn * config.fraud_cost + fp * config.friction_cost)
    axes[1].plot(grid, costs, label=name)
axes[1].set_xlabel("decision threshold")
axes[1].set_ylabel("expected cost")
axes[1].set_title("Cost vs threshold (fraud 500 : friction 5)")
axes[1].legend()
fig.tight_layout()
plt.show()


/var/folders/89/wjdg33k579n81f57cjynwt6h0000gn/T/ipykernel_5320/4120324010.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Does stacking help?

The ensemble combines base-learner probabilities in a regularised logistic
meta-learner. Its meta-features come from **time-ordered out-of-fold**
predictions: fold *k* only ever trains on rows before its hold-out block, so no
future label leaks into the meta-learner. (sklearn's `StackingClassifier` cannot
be used directly here - its internal `cross_val_predict` rejects
`TimeSeriesSplit`.)

In [9]:
from src.models import StackingEnsemble

ensemble = StackingEnsemble(
    ["xgboost", "lightgbm", "catboost"],
    meta_c=1.0, n_splits=5, random_state=42,
)
ensemble.fit(X_train, processed.y_train)
stacked = ensemble.predict_proba(X_valid)[:, 1]

stack_report = evaluate_predictions(
    processed.y_valid, stacked, model_name="stacking",
    figures_dir=PROJECT_ROOT / "outputs" / "figures", make_figures=False,
)
reports["stacking"] = stack_report
print("out-of-fold PR-AUC per base learner:",
      {k: round(v, 4) for k, v in ensemble.oof_scores_.items()})
print(f"stacking  ROC-AUC {stack_report.metrics_at_optimal['roc_auc']:.4f} | "
      f"PR-AUC {stack_report.metrics_at_optimal['pr_auc']:.4f}")


2026-11-06 14:59:32 | INFO     | src.models                   | Stacking OOF fold 1/3: 32,000 train rows, 16,000 holdout rows


2026-11-06 14:59:57 | INFO     | src.models                   | Stacking OOF fold 2/3: 48,000 train rows, 16,000 holdout rows


2026-11-06 15:00:27 | INFO     | src.models                   | Stacking OOF fold 3/3: 64,000 train rows, 16,000 holdout rows


2026-11-06 15:00:27 | INFO     | src.models                   | Stacking OOF PR-AUC [xgboost]: 0.4895


2026-11-06 15:00:27 | INFO     | src.models                   | Stacking OOF PR-AUC [lightgbm]: 0.5049


2026-11-06 15:00:27 | INFO     | src.models                   | Stacking OOF PR-AUC [catboost]: 0.4859


2026-11-06 15:00:48 | INFO     | src.models                   | Stacking fitted: 3 base learners, meta-learner on 48,000 OOF rows, 100.0s


2026-11-06 15:00:49 | INFO     | src.evaluation               | stacking: ROC-AUC=0.8735 PR-AUC=0.4544 F1@opt=0.1169 thr=0.010 cost=71,910


out-of-fold PR-AUC per base learner: {'xgboost': 0.4895, 'lightgbm': 0.5049, 'catboost': 0.4859}
stacking  ROC-AUC 0.8735 | PR-AUC 0.4544


## 7. Why the model fires: SHAP on the best model

Global ranking plus a per-transaction view. The beeswarm shows direction
(colour) as well as magnitude, which is what a fraud analyst needs in order to
trust an alert.

In [10]:
from src.xai import shap_global_explanations

best_name = max(reports, key=lambda k: reports[k].metrics_at_optimal["pr_auc"])
best_model = fitted.get(best_name, ensemble)
print("explaining:", best_name)
xai_report = shap_global_explanations(
    best_model,
    X_valid.sample(n=min(3_000, len(X_valid)), random_state=42),
    output_dir=PROJECT_ROOT / "outputs" / "figures",
    max_display=20,
)
xai_report.figures


2026-11-06 15:00:49 | INFO     | src.xai                      | SHAP will explain the xgboost base learner


2026-11-06 15:00:49 | INFO     | src.xai                      | Computing SHAP values for 3,000 rows


explaining: stacking


2026-11-06 15:00:52 | INFO     | src.xai                      | Top features by SHAP: ['TransactionDT', 'D2', 'D3', 'card1', 'TransactionAmt', 'C13', 'D4', 'C11', 'TransactionDT_log', 'C1']


[PosixPath('/Users/dharun/FFDUEAIAEM/financial_fraud_detection/outputs/figures/shap_summary_beeswarm.png'),
 PosixPath('/Users/dharun/FFDUEAIAEM/financial_fraud_detection/outputs/figures/shap_feature_importance_bar.png')]

## 8. Handoff to the CLI pipeline

This notebook deliberately avoided Optuna (slow) and used a sample. The
production path is:

```bash
# full data, tuned, with explainability
python main.py --data_path data/raw --tune --run_shap_selection --explain

# quick sanity check
python main.py --data_path data/raw --sample_rows 60000 --no-stacking
```

Artifacts land in `outputs/models/`, `outputs/figures/` and `outputs/reports/`.
See `README.md` for the benchmark table.